# Chapter 3: Pydantic — Type-Safe Data Validation

### Why This Matters

Every volume so far has, implicitly, trusted that data arriving into a function or a model is correctly shaped. In practice, data flowing in from an API request, a form submission, a config file, or (as Volume 09 touched on) an LLM's structured output is not automatically trustworthy — it needs to be **validated** before your code relies on it. **Pydantic** uses Python's type annotations (recall the type hints you've seen throughout function signatures) to turn a class definition into an executable, enforced schema.

In [ ]:
from pydantic import BaseModel, Field

class Customer(BaseModel):
    customer_id: int
    name: str
    age: int = Field(ge=18, le=120)     # ge = "greater than or equal", le = "less than or equal" — a valid range
    segment: str

customer = Customer(customer_id=101, name="Asha", age=29, segment="Premium")
print(customer.model_dump())            # convert back to a plain dict

If you tried to create a `Customer` with `age=15` or `age="thirty"` (a string, not an int), Pydantic would immediately raise a clear `ValidationError` explaining exactly which field failed and why — rather than allowing invalid data to silently propagate deeper into your pipeline and cause a confusing failure (or, worse, a wrong but silently "successful" result) much further downstream, where the actual cause would be far harder to trace back.

### 3.1 Nested Models — Validating Real-World Structured Data

In [ ]:
class Item(BaseModel):
    sku: str
    qty: int = Field(gt=0)             # gt = "greater than" (strictly) — zero or negative quantities are invalid
    price: float = Field(gt=0)

class Order(BaseModel):
    order_id: str
    items: list[Item]                    # a LIST of nested Item models — Pydantic validates every element

order = Order(order_id="O-1", items=[{"sku": "P1", "qty": 2, "price": 99.0}])
print(order.model_dump())

Notice `items=[{"sku": "P1", ...}]` was passed as a plain dictionary, and Pydantic automatically converted and validated it into a proper `Item` object nested inside `Order` — this recursive validation of nested structures is precisely what makes Pydantic suited to real-world JSON-shaped data (API payloads, database documents, config files), which is almost never flat.

### 3.2 Why This Matters Specifically for AI/ML Work

Recall Volume 09's LangChain chapter, where forcing an LLM's response into a Pydantic schema was flagged as the standard way to make generative output reliable. The same principle applies to any ML prediction API you build:

In [ ]:
class PredictionResponse(BaseModel):
    customer_id: int
    model_version: str
    churn_probability: float = Field(ge=0.0, le=1.0)     # a probability MUST be between 0 and 1 — enforce it
    predicted_label: str
    top_features: list[str]

Defining this schema once and validating every prediction response against it — both requests coming in and responses going out — catches an entire class of bugs early and explicitly (a model accidentally returning a probability of 1.4 due to a code error gets caught immediately at the schema boundary, rather than silently corrupting a downstream report) and gives any other team consuming your model's API a single, precise, self-documenting contract for exactly what to expect.

### Cheat Sheet — Pydantic

| Task | Code |
|---|---|
| Basic model | `class X(BaseModel): field: type` |
| Constrained field | `field: int = Field(ge=0, le=100)` |
| Nested model | `field: list[OtherModel]` |
| Validate + convert | `Model(**dict_of_data)` |
| Back to a plain dict | `instance.model_dump()` |
| Catch invalid data | `except pydantic.ValidationError as e:` |

---